# Ekstraksi Remote Sensing Bali: zonal stats per kab/kota (Baseline vs Shock)

**Landsat 8** (NDVI, NDWI, NDBI, BUI), **MODIS MOD11A2** (LST), **VIIRS VNP46A2 Black Marble** (NTL), **Sentinel-5P OFFL** (NO2, CO, AER_AI, HCHO).

- **Baseline: Jan-Feb 2020**, **Shock: Apr-Jul 2020** (komposit per periode, median untuk sensor optik/NTL/LST, mean untuk S5P).
- Pembanding tahun 2019 dengan jendela yang sama (opsional) untuk mengoreksi pengaruh musiman.
- Tambahan (opsional): data **bulanan** untuk NTL, LST, dan S5P. Landsat tidak dibuat bulanan karena revisit 16 hari dan tutupan awan musim hujan.


**Output (CSV di `OUT_DIR`)**
- `rs_bali_summary_baseline_vs_shock.csv`: baseline, shock, delta, % perubahan, jumlah piksel valid (long)
- `rs_bali_summary_wide_QGIS.csv`: satu baris per kab/kota, siap di-join ke shp di QGIS
- `rs_bali_monthly_long.csv` dan `rs_bali_monthly_wide_QGIS.csv`: data bulanan (NTL, LST, S5P)

## 1. Konfigurasi

In [ ]:
import os, json, glob, time, zipfile, shutil
import numpy as np
import pandas as pd

# ================= UBAH BAGIAN INI =================
GEE_PROJECT  = "ee-222212522"
NAME_COL     = None
YEAR_MAIN    = 2020
YEAR_CONTROL = 2019

# Jendela komposit untuk YEAR_MAIN. Tanggal akhir EKSKLUSIF (aturan filterDate di GEE).
PERIODS = {
    "baseline": ("2020-01-01", "2020-03-01"),   # Jan-Feb 2020
    "shock":    ("2020-04-01", "2020-08-01"),   # Apr-Jul 2020
}

# Data bulanan tambahan (tidak termasuk Landsat)
RUN_MONTHLY  = True
MONTHLY_VARS = ["NTL", "LST", "NO2", "CO", "AER_AI", "HCHO"]
BASELINE_M, TRANSITION_M, SHOCK_M = [1, 2], [3], [4, 5, 6, 7]

OUT_DIR      = "/content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data"
QUICK_TEST   = False                       # True = tes cepat: hanya komposit Shock tahun utama
# ====================================================

MONTHS = sorted(BASELINE_M + TRANSITION_M + SHOCK_M)
YEARS  = [YEAR_MAIN] + ([YEAR_CONTROL] if YEAR_CONTROL else [])

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUT_DIR, exist_ok=True)
print("Output akan disimpan di:", OUT_DIR)

Mounted at /content/drive
Output akan disimpan di: /content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data


## 2. Login Earth Engine

In [ ]:
import ee
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
print("Earth Engine siap.")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine siap.


## 3. Upload shapefile kab/kota Bali (zip)

In [ ]:
import geopandas as gpd
import shapely.ops
from google.colab import files

print("Upload file .zip yang berisi .shp, .shx, .dbf, .prj")
up = files.upload()
zip_name = list(up.keys())[0]

shutil.rmtree("/content/shp", ignore_errors=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content/shp")

shp_files = glob.glob("/content/shp/**/*.shp", recursive=True)
assert shp_files, "Tidak ada file .shp di dalam zip."
if len(shp_files) > 1:
    print("Ada lebih dari 1 shp, dipakai yang pertama:", shp_files)

gdf = gpd.read_file(shp_files[0])
if gdf.crs is None:
    print("PERINGATAN: CRS tidak terbaca, diasumsikan EPSG:4326.")
    gdf = gdf.set_crs(4326)
else:
    gdf = gdf.to_crs(4326)

print("Kolom atribut:", [c for c in gdf.columns if c != "geometry"])
print(gdf.drop(columns="geometry").head(12))

# --- tentukan kolom nama kab/kota ---
NAME = NAME_COL
if NAME is None:
    upper = {c.upper(): c for c in gdf.columns}
    for cand in ["WADMKK", "NAMOBJ", "KABKOT", "KAB_KOTA", "NAMA_KAB", "KABUPATEN", "NAME_2", "NAMA"]:
        if cand in upper:
            NAME = upper[cand]
            break
if NAME is None or NAME not in gdf.columns:
    raise ValueError("Kolom nama kab/kota tidak ketemu. Isi NAME_COL di sel 1 sesuai daftar kolom di atas.")
print("Kolom nama yang dipakai:", NAME)

# --- rapikan geometri ---
gdf = gdf[[NAME, "geometry"]].copy()
try:
    gdf["geometry"] = gdf.geometry.make_valid()
except Exception:
    gdf["geometry"] = gdf.geometry.buffer(0)
gdf = gdf.dissolve(by=NAME, as_index=False)                      # 1 baris per kab/kota
gdf["geometry"] = gdf.geometry.simplify(0.0002, preserve_topology=True)   # ~20 m, biar payload ringan
gdf["geometry"] = gdf.geometry.apply(lambda g: shapely.ops.transform(lambda x, y, z=None: (x, y), g))  # buang Z
gdf = gdf.rename(columns={NAME: "kab_kota"})

gj = gdf.to_json()
print(f"Jumlah kab/kota: {len(gdf)} | ukuran payload: {len(gj)/1e6:.2f} MB")
if len(gdf) != 9:
    print("PERINGATAN: Bali seharusnya 9 kab/kota. Cek shapefile kamu.")
if len(gj) > 8e6:
    raise ValueError("Payload terlalu besar. Naikkan toleransi simplify (mis. 0.001).")

regions = ee.FeatureCollection(json.loads(gj))
bali_bounds = regions.geometry().bounds()
print("Terbaca di GEE:", regions.size().getInfo(), "fitur")

Upload file .zip yang berisi .shp, .shx, .dbf, .prj


Saving batas_kabkot_bali.zip to batas_kabkot_bali.zip
Kolom atribut: ['KODE_KK', 'KODE_PROV', 'KAB_KOTA', 'PROVINSI', 'FID']
  KODE_KK KODE_PROV       KAB_KOTA PROVINSI    FID
0   51.71        51  Kota Denpasar     Bali  51.71
1   51.04        51        Gianyar     Bali  51.04
2   51.01        51       Jembrana     Bali  51.01
3   51.05        51      Klungkung     Bali  51.05
4   51.08        51       Buleleng     Bali  51.08
5   51.02        51        Tabanan     Bali  51.02
6   51.07        51     Karangasem     Bali  51.07
7   51.03        51         Badung     Bali  51.03
8   51.06        51         Bangli     Bali  51.06
Kolom nama yang dipakai: KAB_KOTA
Jumlah kab/kota: 9 | ukuran payload: 0.31 MB
Terbaca di GEE: 9 fitur


## 4. Definisi sensor



In [ ]:
SENSORS = {
    "NTL":  dict(label="Night-Time Light (VIIRS Black Marble VNP46A2)", type="direct",
                 collection="NASA/VIIRS/002/VNP46A2", band="Gap_Filled_DNB_BRDF_Corrected_NTL",
                 scale=500, scale_factor=0.1, offset=0.0, unit="nW/cm2/sr", method="median",
                 valid_range=(0, 65534)),
    "LST":  dict(label="Land Surface Temperature siang (MODIS MOD11A2)", type="direct",
                 collection="MODIS/061/MOD11A2", band="LST_Day_1km",
                 scale=1000, scale_factor=0.02, offset=-273.15, unit="degC", method="median",
                 valid_range=(7500, 65535)),
    "NDBI": dict(label="NDBI (Landsat 8)", type="landsat", scale=30, unit="indeks", method="median"),
    "NDWI": dict(label="NDWI (Landsat 8)", type="landsat", scale=30, unit="indeks", method="median"),
    "BUI":  dict(label="BUI (Landsat 8)",  type="landsat", scale=30, unit="indeks", method="median"),
    "NDVI": dict(label="NDVI (Landsat 8)", type="landsat", scale=30, unit="indeks", method="median"),
    "NO2":  dict(label="NO2 (Sentinel-5P)", type="direct",
                 collection="COPERNICUS/S5P/OFFL/L3_NO2", band="tropospheric_NO2_column_number_density",
                 scale=3500, scale_factor=1.0, offset=0.0, unit="mol/m2", method="mean"),
    "CO":   dict(label="CO (Sentinel-5P)", type="direct",
                 collection="COPERNICUS/S5P/OFFL/L3_CO", band="CO_column_number_density",
                 scale=7000, scale_factor=1.0, offset=0.0, unit="mol/m2", method="mean"),
    "AER_AI": dict(label="Aerosol Index (Sentinel-5P)", type="direct",
                 collection="COPERNICUS/S5P/OFFL/L3_AER_AI", band="absorbing_aerosol_index",
                 scale=3500, scale_factor=1.0, offset=0.0, unit="indeks", method="mean"),
    "HCHO": dict(label="HCHO (Sentinel-5P)", type="direct",
                 collection="COPERNICUS/S5P/OFFL/L3_HCHO", band="tropospheric_HCHO_column_number_density",
                 scale=3500, scale_factor=1.0, offset=0.0, unit="mol/m2", method="mean"),
}
LANDSAT_VARS = ["NDVI", "NDWI", "NDBI", "BUI"]
print(f"{len(SENSORS)} variabel dikonfigurasi:", ", ".join(SENSORS))

# ---------- helper tanggal ----------
def shift_year(datestr, delta):
    return (pd.Timestamp(datestr) + pd.DateOffset(years=delta)).strftime("%Y-%m-%d")

def period_windows(year):
    d = year - YEAR_MAIN
    return {k: (shift_year(s, d), shift_year(e, d)) for k, (s, e) in PERIODS.items()}

def group_by_scale(vars_):
    g = {}
    for v in vars_:
        g.setdefault(SENSORS[v]["scale"], []).append(v)
    return g

def build_jobs():
    """Daftar pekerjaan: (mode, scale, year, label, start, end, vars). Tahun utama dikerjakan lebih dulu."""
    jobs = []
    for scale, vs in group_by_scale(list(SENSORS)).items():
        for y in YEARS:
            for label, (s, e) in period_windows(y).items():
                jobs.append(("period", scale, y, label, s, e, vs))
    if RUN_MONTHLY:
        for scale, vs in group_by_scale(MONTHLY_VARS).items():
            for y in YEARS:
                for m in MONTHS:
                    s = f"{y}-{m:02d}-01"
                    e = (pd.Timestamp(s) + pd.DateOffset(months=1)).strftime("%Y-%m-%d")
                    jobs.append(("month", scale, y, f"{y}-{m:02d}", s, e, vs))
    jobs.sort(key=lambda j: (j[2] != YEAR_MAIN, j[0] != "period"))   # stabil: tahun utama & komposit periode dulu
    return jobs

# ---------- builder gambar ----------
def empty_img(bands):
    """Gambar kosong (semua piksel masked) supaya periode tanpa data menghasilkan NaN, bukan error."""
    return ee.Image.constant([0] * len(bands)).rename(bands).selfMask()

def safe(col, fn, bands):
    return ee.Image(ee.Algorithms.If(col.size().gt(0), fn(col), empty_img(bands)))

def direct_image(v, start, end):
    cfg = SENSORS[v]
    col = (ee.ImageCollection(cfg["collection"]).filterBounds(bali_bounds)
           .filterDate(start, end).select(cfg["band"]))
    vr = cfg.get("valid_range")
    def conv(i):
        if vr:
            i = i.updateMask(i.gte(vr[0]).And(i.lte(vr[1])))     # buang nilai fill
        return i.multiply(cfg["scale_factor"]).add(cfg["offset"])
    def agg(c):
        c = c.map(conv)
        r = c.median() if cfg["method"] == "median" else c.mean()
        return r.rename(v)
    return safe(col, agg, [v])

def landsat_image(start, end):
    col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
           .filterBounds(bali_bounds).filterDate(start, end))
    def prep(img):
        qa = img.select("QA_PIXEL")
        bad = qa.bitwiseAnd(31).neq(0)     # bit 0-4: fill, dilated cloud, cirrus, cloud, cloud shadow
        r = (img.select(["SR_B3", "SR_B4", "SR_B5", "SR_B6"])
             .multiply(0.0000275).add(-0.2).updateMask(bad.Not()))
        ndvi = r.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI")
        ndwi = r.normalizedDifference(["SR_B3", "SR_B5"]).rename("NDWI")
        ndbi = r.normalizedDifference(["SR_B6", "SR_B5"]).rename("NDBI")
        bui  = ndbi.subtract(ndvi).rename("BUI")
        return ee.Image.cat([ndvi, ndwi, ndbi, bui])
    return safe(col, lambda c: c.map(prep).median(), LANDSAT_VARS)

def build_group_image(vars_, start, end):
    imgs = []
    lv = [v for v in vars_ if SENSORS[v]["type"] == "landsat"]
    if lv:
        imgs.append(landsat_image(start, end).select(lv))
    for v in vars_:
        if SENSORS[v]["type"] == "direct":
            imgs.append(direct_image(v, start, end))
    return ee.Image.cat(imgs)

REDUCER = ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True)
print("Builder siap.")

10 variabel dikonfigurasi: NTL, LST, NDBI, NDWI, BUI, NDVI, NO2, CO, AER_AI, HCHO
Builder siap.


### Cek jumlah scene Landsat 8 per jendela
Kalau jumlahnya sangat sedikit (mis. 1-2 scene), komposit banyak piksel kosong karena awan. Lihat juga kolom `n_pixel` di output.

In [ ]:
for y in YEARS:
    for label, (s, e) in period_windows(y).items():
        n = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(bali_bounds)
             .filterDate(s, e).size().getInfo())
        print(f"Landsat 8 | {label:8s} {y} ({s} s.d. <{e}): {n} scene")

Landsat 8 | baseline 2020 (2020-01-01 s.d. <2020-03-01): 10 scene
Landsat 8 | shock    2020 (2020-04-01 s.d. <2020-08-01): 23 scene
Landsat 8 | baseline 2019 (2019-01-01 s.d. <2019-03-01): 11 scene
Landsat 8 | shock    2019 (2019-04-01 s.d. <2019-08-01): 24 scene


## 5. Ekstraksi (dengan checkpoint)
Hasil disimpan ke Drive setiap satu pekerjaan selesai, jadi kalau runtime putus tinggal dijalankan ulang.

In [ ]:
def with_retry(fn, tries=3, wait=10):
    for i in range(tries):
        try:
            return fn()
        except Exception as e:
            print(f"    percobaan {i+1} gagal: {str(e)[:200]}")
            time.sleep(wait * (i + 1))
    return None

def get_stat(p, var, stat, default=np.nan):
    """Nama properti hasil reduceRegions bisa 'NDVI_mean' (multi-band) atau 'mean' (1 band)."""
    for k in (f"{var}_{stat}", stat):
        if k in p and p[k] is not None:
            return p[k]
    return default

CKPT = f"{OUT_DIR}/_checkpoint_test.csv" if QUICK_TEST else f"{OUT_DIR}/_checkpoint_v2.csv"
if os.path.exists(CKPT):
    prev = pd.read_csv(CKPT, dtype={"label": str})
    rows = prev.to_dict("records")
    done = set(zip(prev["mode"], prev["scale"], prev["year"], prev["label"]))
    print(f"Checkpoint ditemukan: {len(done)} pekerjaan sudah selesai, dilewati.")
else:
    rows, done = [], set()

jobs = build_jobs()
if QUICK_TEST:
    jobs = [j for j in jobs if j[0] == "period" and j[2] == YEAR_MAIN and j[3] == "shock"]
print(f"Total pekerjaan: {len(jobs)}")

failed, first = [], True
for i, (mode, scale, y, label, s, e, vs) in enumerate(jobs, 1):
    if (mode, scale, y, label) in done:
        continue
    t0 = time.time()
    img = build_group_image(vs, s, e)
    fc = img.reduceRegions(collection=regions, reducer=REDUCER, scale=scale, tileScale=4)
    fc = fc.map(lambda f: ee.Feature(None, f.toDictionary()))       # buang geometri, ambil atribut saja
    feats = with_retry(lambda: fc.getInfo()["features"])
    if feats is None:
        failed.append((mode, scale, y, label))
        print(f"[{i}/{len(jobs)}] {mode} {label} {y} skala {scale} m: GAGAL")
        continue
    if first:
        print("Contoh nama properti hasil:", list(feats[0]["properties"].keys()))
        first = False
    for f in feats:
        p = f["properties"]
        for v in vs:
            rows.append({"mode": mode, "scale": scale, "year": y, "label": label,
                         "kab_kota": p["kab_kota"], "var": v,
                         "mean": get_stat(p, v, "mean"),
                         "n_pixel": get_stat(p, v, "count", default=0)})
    pd.DataFrame(rows).to_csv(CKPT, index=False)
    print(f"[{i}/{len(jobs)}] {mode} {label} {y} | {','.join(vs)} | {time.time()-t0:.0f} dtk")

if failed:
    print("\nYang gagal (jalankan ulang sel ini untuk mencoba lagi):", failed)
else:
    print("\nSemua selesai.")

Total pekerjaan: 76
Contoh nama properti hasil: ['count', 'kab_kota', 'mean']
[1/76] period baseline 2020 | NTL | 4 dtk
[2/76] period shock 2020 | NTL | 5 dtk
[3/76] period baseline 2020 | LST | 1 dtk
[4/76] period shock 2020 | LST | 1 dtk
[5/76] period baseline 2020 | NDBI,NDWI,BUI,NDVI | 6 dtk
[6/76] period shock 2020 | NDBI,NDWI,BUI,NDVI | 6 dtk
[7/76] period baseline 2020 | NO2,AER_AI,HCHO | 3 dtk
[8/76] period shock 2020 | NO2,AER_AI,HCHO | 6 dtk
[9/76] period baseline 2020 | CO | 5 dtk
[10/76] period shock 2020 | CO | 7 dtk
[11/76] month 2020-01 2020 | NTL | 1 dtk
[12/76] month 2020-02 2020 | NTL | 3 dtk
[13/76] month 2020-03 2020 | NTL | 2 dtk
[14/76] month 2020-04 2020 | NTL | 2 dtk
[15/76] month 2020-05 2020 | NTL | 2 dtk
[16/76] month 2020-06 2020 | NTL | 2 dtk
[17/76] month 2020-07 2020 | NTL | 2 dtk
[18/76] month 2020-01 2020 | LST | 1 dtk
[19/76] month 2020-02 2020 | LST | 1 dtk
[20/76] month 2020-03 2020 | LST | 1 dtk
[21/76] month 2020-04 2020 | LST | 1 dtk
[22/76] month

## 6. Buat CSV output
- `delta` = Shock - Baseline. Untuk **NDVI, NDWI, NDBI, BUI, LST, AER_AI**, pakai `delta` (selisih absolut), karena persen perubahan tidak bermakna untuk indeks yang bisa mendekati nol atau negatif. Untuk **NTL, NO2, CO, HCHO**, `pct` lebih intuitif.
- `dadj` dan `padj` = delta (atau pct) 2020 dikurangi delta (atau pct) 2019: koreksi musiman, yaitu sisa perubahan yang tidak dijelaskan pergantian musim hujan ke kemarau.
- `nb` dan `ns` = jumlah piksel valid pada baseline dan shock. Nilai kecil berarti rata-rata kab/kota itu kurang bisa diandalkan.
- Di file wide, `AER_AI` ditulis `AER` supaya nama kolom tidak lebih dari 10 karakter.

In [ ]:
df = pd.DataFrame(rows)
assert len(df) > 0, "Belum ada data. Jalankan sel 5 dulu."
UNITS = {v: SENSORS[v]["unit"] for v in SENSORS}
df["unit"] = df["var"].map(UNITS)

WIDE_NAME = {"AER_AI": "AER"}
wn = lambda v: WIDE_NAME.get(v, v)

def period_of(m):
    return "baseline" if m in BASELINE_M else ("transisi" if m in TRANSITION_M else "shock")

KEYS = ["kab_kota", "var", "year", "label"]

# ================= A) Ringkasan komposit Baseline vs Shock =================
per = df[df["mode"] == "period"].drop_duplicates(KEYS, keep="last")
summary = None
if len(per):
    pm = per.pivot(index=["kab_kota", "var", "year"], columns="label", values=["mean", "n_pixel"])
    pm.columns = [f"{a}_{b}" for a, b in pm.columns]
    pm = pm.reset_index().rename(columns={"mean_baseline": "baseline", "mean_shock": "shock",
                                          "n_pixel_baseline": "nb", "n_pixel_shock": "ns"})
    for c in ["baseline", "shock", "nb", "ns"]:
        if c not in pm.columns:
            pm[c] = np.nan
    pm["delta"] = pm["shock"] - pm["baseline"]
    pm["pct"] = (pm["delta"] / pm["baseline"].abs() * 100).replace([np.inf, -np.inf], np.nan)

    summary = pm[pm["year"] == YEAR_MAIN].drop(columns="year").copy()
    has_ctrl = bool(YEAR_CONTROL) and (pm["year"] == YEAR_CONTROL).any()
    if has_ctrl:
        ctrl = (pm[pm["year"] == YEAR_CONTROL][["kab_kota", "var", "baseline", "shock", "delta", "pct"]]
                .rename(columns=lambda c: c if c in ("kab_kota", "var") else f"{c}_{YEAR_CONTROL}"))
        summary = summary.merge(ctrl, on=["kab_kota", "var"], how="left")
        summary["dadj"] = summary["delta"] - summary[f"delta_{YEAR_CONTROL}"]
        summary["padj"] = summary["pct"] - summary[f"pct_{YEAR_CONTROL}"]
    summary["unit"] = summary["var"].map(UNITS)
    summary = summary.sort_values(["var", "kab_kota"]).reset_index(drop=True)
    summary.to_csv(f"{OUT_DIR}/rs_bali_summary_baseline_vs_shock.csv", index=False)

    sw = summary.rename(columns={"baseline": "base"}).copy()
    sw["var"] = sw["var"].map(wn)
    val_cols = ["base", "shock", "delta", "pct"] + (["dadj", "padj"] if has_ctrl else []) + ["nb", "ns"]
    swide = sw.pivot(index="kab_kota", columns="var", values=val_cols)
    swide.columns = [f"{v}_{c}" for c, v in swide.columns]     # mis. NDVI_delta, NTL_pct
    swide.reset_index().to_csv(f"{OUT_DIR}/rs_bali_summary_wide_QGIS.csv", index=False)

# ================= B) Data bulanan (NTL, LST, S5P) =================
mon = df[df["mode"] == "month"].drop_duplicates(KEYS, keep="last").copy()
if len(mon):
    mon["month"] = mon["label"].str[-2:].astype(int)
    mon["period"] = mon["month"].map(period_of)
    monthly = (mon[["kab_kota", "var", "year", "month", "period", "mean", "n_pixel", "unit"]]
               .sort_values(["var", "kab_kota", "year", "month"]).reset_index(drop=True))
    monthly.to_csv(f"{OUT_DIR}/rs_bali_monthly_long.csv", index=False)
    key = (mon["var"].map(wn) + "_" + (mon["year"] % 100).astype(str).str.zfill(2)
           + mon["month"].astype(str).str.zfill(2))            # mis. NTL_2004 = NTL April 2020
    mwide = mon.assign(key=key).pivot(index="kab_kota", columns="key", values="mean").reset_index()
    mwide.columns.name = None
    mwide.to_csv(f"{OUT_DIR}/rs_bali_monthly_wide_QGIS.csv", index=False)

print("File tersimpan di", OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    if not f.startswith("_"):
        print("  -", f)

# ---- cek ketersediaan data ----
chk = per.assign(kosong=per["mean"].isna())
print("\nKomposit periode: jumlah kab/kota dengan nilai kosong (NaN) dan piksel valid minimum")
print(chk.groupby(["var", "year", "label"]).agg(kosong=("kosong", "sum"), n_min=("n_pixel", "min")).to_string())
summary.head(20) if summary is not None else None

File tersimpan di /content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data
  - 01. Data_IRIO_17_Siap.xlsx
  - 02. Ekstraksi_Data.xlsx
  - 03. Proporsi_Shock_Pariwisata_2020.xlsx
  - 04. Hasil_Evaluasi_Indikator_2020.xlsx
  - 05. Hasil_Shock_FinalDemand_Bali_2025.xlsx
  - Data_Skripsi_New.xlsx
  - Direct Flight_Data.xlsx
  - Ekstraksi_Data.gsheet
  - Konstruksi Shock
  - Pengambila Data
  - SHP
  - Simulasi Shock
  - Spatial Analisis
  - matriks_konektivitas_spasial_34x34.xlsx
  - raw_per_kategori
  - rs_bali_monthly_long.csv
  - rs_bali_monthly_wide_QGIS.csv
  - rs_bali_summary_baseline_vs_shock.csv
  - rs_bali_summary_wide_QGIS.csv

Komposit periode: jumlah kab/kota dengan nilai kosong (NaN) dan piksel valid minimum
                      kosong   n_min
var    year label                   
AER_AI 2019 baseline       0      20
            shock          0      20
       2020 baseline       0      20
            shock          0      20
BUI    2019 baseline       0  144011


,kab_kota,var,baseline,shock,nb,ns,delta,pct,baseline_2019,shock_2019,delta_2019,pct_2019,dadj,padj,unit
0,Badung,AER_AI,-1.416953,-1.565232,66.0,66.0,-0.148279,-10.464649,-1.152509,-1.297632,-0.145123,-12.591928,-0.003156,2.127279,indeks
1,Bangli,AER_AI,-1.034626,-1.406710,65.0,65.0,-0.372084,-35.963145,-0.783769,-1.137714,-0.353945,-45.159364,-0.018139,9.196219,indeks
2,Buleleng,AER_AI,-1.270088,-1.403304,160.0,160.0,-0.133216,-10.488712,-1.009753,-1.161037,-0.151284,-14.982271,0.018068,4.493558,indeks
3,Gianyar,AER_AI,-1.452964,-1.513659,48.0,48.0,-0.060695,-4.177334,-1.113705,-1.307520,-0.193814,-17.402666,0.133119,13.225331,indeks
4,Jembrana,AER_AI,-1.355630,-1.221618,99.0,99.0,0.134012,9.885558,-1.105298,-0.945480,0.159819,14.459338,-0.025807,-4.573779,indeks
5,Karangasem,AER_AI,-1.105819,-1.615260,93.0,93.0,-0.509441,-46.069127,-0.841463,-1.315879,-0.474415,-56.379763,-0.035026,10.310636,indeks
6,Klungkung,AER_AI,-1.529717,-1.746839,46.0,46.0,-0.217121,-14.193568,-1.223633,-1.453050,-0.229417,-18.748813,0.012295,4.555245,indeks
7,Kota Denpasar,AER_AI,-1.384332,-1.688635,20.0,20.0,-0.304303,-21.981954,-1.185736,-1.419072,-0.233336,-19.678618,-0.070967,-2.303336,indeks
8,Tabanan,AER_AI,-1.417628,-1.254675,95.0,95.0,0.162953,11.494757,-1.142234,-0.968901,0.173333,15.174901,-0.010380,-3.680145,indeks
9,Badung,BUI,-0.932696,-0.949776,418284.0,455442.0,-0.017080,-1.831231,-0.984516,-0.907949,0.076567,7.777095,-0.093647,-9.608327,indeks
